In [ ]:
import torch
from collections import Counter
import tensorflow as tf
print('TF version:', tf.__version__)
print("Current PyTorch version:", torch.__version__)
print('Cuda Version: ', torch.version.cuda)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
from Shuffling_functions import flow_features, PacketSlicer, RangeNumbers, first_packets_equal,packet_sizes_random, delay_calc, last_packets_shuffle
from Shuffling_functions import convert_to_ms, intervals_random, dl_delay_eq
from Classification import multi_web_service_label
import bisect
import random


In [ ]:
import cryptography
print(cryptography.__version__)

In [ ]:
def sizes_discr(row):
    for i in range(len(row['Sizes'])):
        if (row['Is_uplink'][i] == 0 or row['Is_uplink'][i] == 1):
            n = row['Sizes'][i]
            sizes = [500, 700, 1200]
            greater_numbers = [x for x in sizes if x > n]
            if not greater_numbers: 
                row['Sizes'][i] = 1200 
            else:
                row['Sizes'][i] = min(greater_numbers, key=lambda x: abs(x - n))
    return row

In [ ]:
def sizes_equal(row):
    for i in range(len(row['Sizes'])):
        if(row['Sizes'][i]<700 and row['Is_uplink'][i] == 0):
            row['Sizes'][i] = 700
    return row

In [ ]:
import pandas as pd
import numpy as np
from time import time
from tqdm import tqdm
import re

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import f1_score
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

from classification_reports_UW import classfication_report, misclassifications, misclassification_pairs, short_report
from classification_reports_UW import classfication_report_df_save, classification_reports_average, feature_importance_average
from input_run_UW import labels_to_int, labels_to_tensor, mapdict_from_label_names
from input_run_UW import payload_data_exctraction, payload_data_and_TSPS_exctraction, payload_alignment
from input_run_UW import MATEC_data, BGRUA_data, ABRF_data, RBRF_data, hypridC45_data
from input_run_UW import hRBRF_data, hABRF_data
from input_run_UW import bag_of_flows, CH_SH_vector_from_table

from input_run_UW import  RF_feature_selection_better_than_random, parameters_position

from input_run_UW import train_validation_test_split, TSPS_features_from_Table, TSPS_handshake_httpReq
# from QUIC_parser import quic_payload_data_exctraction, quic_payload_and_TSPS_data_exctraction
# from QUIC_parser import quic_payload_data_TSPS_IPdst_exctraction, quic_payload_data_TSPShandshake_exctraction
from input_run_UW import payload_data_TSPS_IPdst_exctraction, flow_start_duration
from input_run_UW import multi_web_service_label, DataPayload_DataFlow_DataIP_split, DataPayload_DataFlow_split
from Classifiers import BiGRUA_Classifier, MATEC_Classifier
from input_run_UW import majority_voting, majority_voting_with_time
from input_run_UW import ip_distinction_features, get_full_timestamp_pd, bag_of_flows_TSPS


from input_run_UW import SH_dict, CH_dict, ipVector, TSPS_features_from_Table, payload_data_TSPShandshake_exctraction
from input_run_UW import payload_data_TSPShandshake_BoF_exctraction, BoF_features, JA3_and_JA3S_fingerprint
from input_run_UW import CH_SH_recomp_vector_from_table, RBRF_data_without_GREASE, TLS_distinction_metric
#from QUIC_parser import quic_payload_data_TSPShandshake_BoF_exctraction
top = ''
from joblib import Parallel, delayed
from tqdm import tqdm 

In [ ]:
class Early_TC_flow:
    def __init__(self, df):
        self.df = df
        self._prepare_data()
    
    def _prepare_data(self):
        self.df['IAT'] = self.df['Time'].diff().fillna(0)
        self.df['isUplink'] = self.df['isUplink'].replace(0, -1)

    def get_ps_stats_feature_vector(self, packet_limit=30):
        # Ensure each column has 30 values
        def adjust_zero(values, packet_limit=30):
            if len(values) > packet_limit:
                return values[:packet_limit]
            elif len(values) < packet_limit:
                return np.pad(values, (0, packet_limit - len(values)), 'constant')
            return values

        # Standardize IAT and Size
        def standardize(values):
            mean = np.mean(values)
            std = np.std(values)
            return (values - mean) / (std if std > 0 else 1)

        self.df['IAT_standardized'] = standardize(self.df['IAT'])
        self.df['Size_standardized'] = standardize(self.df['Size'])

        iat = adjust_zero(self.df['IAT_standardized'].values, packet_limit)
        size = adjust_zero(self.df['Size_standardized'].values, packet_limit)
        is_uplink = adjust_zero(self.df['isUplink'].values, packet_limit)
        
        del self.df['IAT_standardized'], self.df['Size_standardized']

        # Combine into a single feature vector
        feature_vector = np.concatenate((iat, size, is_uplink))

        return feature_vector

    def get_flow_stat_features_CESNET(self):
        # Calculate features
        sum_size_uplink = sum(self.df[self.df['isUplink'] == 1]['Size'])
        sum_size_downlink = sum(self.df[self.df['isUplink'] == -1]['Size'])
        num_packets_uplink = self.df[self.df['isUplink'] == 1].shape[0]
        num_packets_downlink = self.df[self.df['isUplink'] == -1].shape[0]
        num_roundtrips = ((self.df['isUplink'].shift() != self.df['isUplink']).sum() - 1) 
        last_time_uplink = self.df[self.df['isUplink'] == 1]['Time'].max() if not self.df[self.df['isUplink'] == 1].empty else 0
        last_time_downlink = self.df[self.df['isUplink'] == -1]['Time'].max() if not self.df[self.df['isUplink'] == -1].empty else 0

        features = [sum_size_uplink, sum_size_downlink, num_packets_uplink, 
                    num_packets_downlink, num_roundtrips, last_time_uplink, last_time_downlink]

        return features
    
    def get_flow_stat_features_UW(self):
        # Calculate features
        sum_size_uplink = sum(self.df[self.df['isUplink'] == 1]['Size'])
        sum_size_downlink = sum(self.df[self.df['isUplink'] == -1]['Size'])
        min_size_uplink = np.min(self.df[self.df['isUplink'] == 1]['Size'])
        min_size_downlink = np.min(self.df[self.df['isUplink'] == -1]['Size'])
        max_size_uplink = np.max(self.df[self.df['isUplink'] == 1]['Size'])
        max_size_downlink = np.max(self.df[self.df['isUplink'] == -1]['Size'])
        std_size_uplink = np.std(self.df[self.df['isUplink'] == 1]['Size'])
        std_size_downlink = np.std(self.df[self.df['isUplink'] == -1]['Size'])
        median_size_uplink = np.median(self.df[self.df['isUplink'] == 1]['Size'])
        median_size_downlink = np.median(self.df[self.df['isUplink'] == -1]['Size'])
        
        min_iat_uplink = np.min(self.df[self.df['isUplink'] == 1]['IAT'])
        min_iat_downlink = np.min(self.df[self.df['isUplink'] == -1]['IAT'])
        max_iat_uplink = np.max(self.df[self.df['isUplink'] == 1]['IAT'])
        max_iat_downlink = np.max(self.df[self.df['isUplink'] == -1]['IAT'])
        std_iat_uplink = np.std(self.df[self.df['isUplink'] == 1]['IAT'])
        std_iat_downlink = np.std(self.df[self.df['isUplink'] == -1]['IAT'])
        median_iat_uplink = np.median(self.df[self.df['isUplink'] == 1]['IAT'])
        median_iat_downlink = np.median(self.df[self.df['isUplink'] == -1]['IAT'])
        
        num_packets_uplink = self.df[self.df['isUplink'] == 1].shape[0]
        num_packets_downlink = self.df[self.df['isUplink'] == -1].shape[0]
        num_roundtrips = ((self.df['isUplink'].shift() != self.df['isUplink']).sum() - 1) 
        last_time_uplink = self.df[self.df['isUplink'] == 1]['Time'].max() if not self.df[self.df['isUplink'] == 1].empty else 0
        last_time_downlink = self.df[self.df['isUplink'] == -1]['Time'].max() if not self.df[self.df['isUplink'] == -1].empty else 0

        features = [sum_size_uplink, sum_size_downlink, 
                    min_size_uplink, min_size_downlink,
                    max_size_uplink, max_size_downlink,
                    std_size_uplink, std_size_downlink,
                    median_size_uplink, median_size_downlink, 
                    min_iat_uplink, min_iat_downlink,
                    max_iat_uplink, max_iat_downlink,
                    std_iat_uplink, std_iat_downlink,
                    median_iat_uplink, median_iat_downlink, 
                    num_packets_uplink, num_packets_downlink, 
                    num_roundtrips, last_time_uplink, last_time_downlink]

        return features

In [ ]:
def Luxemburg_Features(Dataset: pd.DataFrame, packet_limit=30, n_jobs=-1) -> [np.array, np.array, np.array]:
    Dataset.index = range(len(Dataset))
    Labels = Dataset['labels']
    
    def process_row(row):
        df = pd.DataFrame({'Time': row['Times'], 'Size': row['Sizes'], 'isUplink': row['Is_uplink']}) 
        Flow_df = Early_TC_flow(df)
        return Flow_df.get_ps_stats_feature_vector(packet_limit), Flow_df.get_flow_stat_features_UW()
    
    results = Parallel(n_jobs=n_jobs)(
    delayed(process_row)(Dataset.iloc[i]) for i in tqdm(Dataset.index))
    
    packet_stat_features, flow_stat_features = zip(*results)
    
    return np.array(packet_stat_features, dtype=float), np.array(flow_stat_features, dtype=float), Labels

In [ ]:
def PacketSlicer(row, N):
    row['Sizes'] = row['Sizes'].strip('[]').split()[:N]
    row['Sizes'] = [int(num) for num in row['Sizes']]
    row['Is_uplink'] = row['Is_uplink'].strip('[]').split()[:N]
    row['Is_uplink'] = [int(num) for num in row['Is_uplink']]
    row['Times'] = row['Times'].strip('[]').split()[:N]
    row['Times'] = [float(num) for num in row['Times']]
    return row

In [ ]:
FLOW_STAT_PATH = ""
PAYLOAD_PATH = ""

In [ ]:
TSPS = pd.read_csv(FLOW_STAT_PATH)
TSPS = TSPS.dropna()
TSPS = TSPS.apply(lambda row: PacketSlicer(row, int(row['N'])), axis=1)
# TSPS = TSPS.apply(lambda row: sizes_discr(row), axis=1)
# TSPS = TSPS.apply(lambda row: sizes_equal(row), axis=1)
TSPS = TSPS.reset_index(drop = True)
TSPS

In [ ]:
SNI = [TSPS.loc[i,'SNI'] for i in range(len(TSPS))]
sni_list = ['hireserve.com','www.oxilion.nl', 'artsandculture.google.com', 'www.zellis.com', 'planogram.scania.com', 'cec.konicaminolta.eu', 'cpireland.crowneplaza.com', 'coptrz.com', 'krystal.io', 'eduspot.co.uk']
TSPS =  TSPS[TSPS['SNI'].isin(sni_list)]
TSPS =  TSPS[TSPS['L4_protocol'] == 'QUIC']
TSPS = TSPS.reset_index(drop=True)
TSPS['labels'] = TSPS['SNI']


In [ ]:
df = pd.read_csv(PAYLOAD_PATH)
df = df[df['pcap_name'].isin(TSPS['pcap_name'])]
df = df.reset_index(drop=True)
TSPS = TSPS[TSPS['pcap_name'].isin(df['pcap_name'])]
TSPS = TSPS.reset_index(drop=True)
TSPS


In [ ]:
def parse_flow(s):
    if isinstance(s, str):
        try:
            return np.array([int(x) for x in s.split(' ') if x.strip() != ''], dtype=np.uint8)
        except Exception:
            return np.array([], dtype=np.uint8)
    return np.array([], dtype=np.uint8)

In [ ]:
df["Flow"] = df["Flow"].apply(parse_flow)

In [ ]:
CH_SH_data = np.array(df['Flow'].tolist())
CH_SH_data

In [ ]:
TimeSeries_data, Flow_Stat_data, Labels  = Luxemburg_Features(TSPS)

In [ ]:
TimeSeries_data = np.nan_to_num(TimeSeries_data)
Flow_Stat_data = np.nan_to_num(Flow_Stat_data)
CH_SH_data = np.nan_to_num(CH_SH_data)

In [ ]:

TLS_data = CH_SH_data
TimeSeries_Features = np.array(TimeSeries_data)
Flow_Stat_Features = np.nan_to_num(np.array(Flow_Stat_data), nan=0.0)
np.array(TLS_data).shape

# Read and transform data to tensor

In [ ]:
def standartised_Fstat(stat_array: np.array) -> np.array:
    return (np.clip(stat_array, 0, np.percentile(stat_array, 95))  - np.median(stat_array))/np.percentile(stat_array, 95)

In [ ]:
TLS_data = np.array(TLS_data)
labels = np.array(TSPS['labels'])

mapdict = mapdict_from_label_names(Labels)
print(mapdict)
for i in range(Flow_Stat_Features.shape[-1]):
        Flow_Stat_Features[:, i] = standartised_Fstat(Flow_Stat_Features[:, i])
Flow_Stat_Features = np.nan_to_num(np.array(Flow_Stat_data), nan=0.0)

X_tls = torch.from_numpy(np.array(CH_SH_data, dtype = np.float32))
X_pstats = torch.from_numpy(np.array(TimeSeries_Features, dtype = np.float32))
X_flowstats = torch.from_numpy(np.array(Flow_Stat_Features, dtype = np.float32))
labels_tensor = torch.tensor([mapdict[label] for label in Labels], dtype=torch.long)

In [ ]:
# Split the data into training+validation and test sets
X_tls_train_val, X_tls_test, X_pstats_train_val, X_pstats_test, X_flowstats_train_val, X_flowstats_test, y_train_val, y_test = train_test_split(
    X_tls, X_pstats, X_flowstats, labels_tensor, test_size=0.2, random_state=42, stratify=labels_tensor
)

# Split the training+validation set into training and validation sets
X_tls_train, X_tls_val, X_pstats_train, X_pstats_val, X_flowstats_train, X_flowstats_val, y_train, y_val = train_test_split(
    X_tls_train_val, X_pstats_train_val, X_flowstats_train_val, y_train_val, test_size=0.1, random_state=42, stratify=y_train_val
)

# Create TensorDataset and DataLoader for training, validation, and test sets
batch_size = 64
train_dataset = TensorDataset(X_tls_train, X_pstats_train, X_flowstats_train, y_train)
val_dataset   = TensorDataset(X_tls_val,   X_pstats_val,   X_flowstats_val,   y_val)
test_dataset  = TensorDataset(X_tls_test,  X_pstats_test,  X_flowstats_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# UW model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UWTrafficClassificationModel(nn.Module):
    def __init__(self, num_classes):
        super(UWTrafficClassificationModel, self).__init__()
        
        # TLS Handshake Bytes Convolutions
        self.tls_conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=2, stride=1, padding = 0),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=2, stride=1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=2, stride=1, padding = 0),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Conv1d(in_channels=64, out_channels=4, kernel_size=2, stride=1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm1d(4),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        
        # Flow Time-series Dense Layer before LSTM
        self.flow_dense_before_lstm = nn.Linear(90, 256)
        
        # Flow Time-series LSTM
        self.flow_lstm = nn.LSTM(input_size=256, hidden_size=256, num_layers=3, batch_first=True)
        
        # Dropout layer after LSTM
        self.lstm_dropout = nn.Dropout(p=0.5)
        
        # Standard Flow Statistics Dense Layers
        self.flow_dense = nn.Sequential(
            nn.Linear(23, 512),
            nn.ReLU(),
            nn.Linear(512, 200),
            nn.ReLU(),
            nn.Linear(200, 200),
            nn.ReLU()
        )
        
        # Combined Dense Layers
        self.combined_dense = nn.Sequential(
            nn.Linear(4*128 + 256 + 200, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, tls_handshake_bytes, flow_time_series, flow_stats):
        # TLS Handshake Bytes
        tls_handshake_bytes = tls_handshake_bytes.view(tls_handshake_bytes.size(0), 1, -1)
        tls_features = self.tls_conv(tls_handshake_bytes)
        tls_features = tls_features.view(tls_features.size(0), -1)  # Flatten
        
        # Flow Time-series
        flow_time_series = F.relu(self.flow_dense_before_lstm(flow_time_series))
        flow_time_series = flow_time_series.unsqueeze(1)  # Add a sequence dimension
        lstm_out, (h_n, c_n) = self.flow_lstm(flow_time_series)
        flow_lstm_features = self.lstm_dropout(h_n[-1])  # Apply dropout to the last hidden state
        
        # Standard Flow Statistics
        flow_stats_features = self.flow_dense(flow_stats)
        
        # Concatenate all features
        combined_features = torch.cat((tls_features, flow_lstm_features, flow_stats_features), dim=1)
        
        # Pass through combined dense layers
        output = self.combined_dense(combined_features)
        
        return output

# Example usage
num_classes = 10  # Replace with the actual number of classes
model = UWTrafficClassificationModel(num_classes)

# Assuming `tls_handshake_bytes`, `flow_time_series`, and `flow_stats` are your input tensors
# tls_handshake_bytes: shape (batch_size, 512)
# flow_time_series: shape (batch_size, 90)
# flow_stats: shape (batch_size, 21)
tls_handshake_bytes = torch.randn(8, 512)
flow_time_series = torch.randn(8, 90)
flow_stats = torch.randn(8, 23)

output = model(tls_handshake_bytes, flow_time_series, flow_stats)
print(output.shape)  # Should be (batch_size, num_classes)


In [ ]:
def train_UW_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_path, plt_save_path):
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    train_f1_scores = []
    val_f1_scores = []

    for epoch in range(num_epochs):
        start_epoch_time = time()
        model.train()
        running_loss = 0.0
        all_train_labels = []
        all_train_predictions = []
        
        for i, (tls, pstats, flowstats, labels) in enumerate(train_loader):
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(tls.to(device), pstats.to(device), flowstats.to(device))
            loss = criterion(outputs, labels.to(device))
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            all_train_labels.extend(labels.cpu().numpy())
            all_train_predictions.extend(predicted.cpu().numpy())
        
        train_f1_score_epoch = f1_score(all_train_labels, all_train_predictions, average='macro')
        train_losses.append(running_loss / len(train_loader))
        train_f1_scores.append(train_f1_score_epoch)
        
        # Validation step
        model.eval()
        val_loss = 0.0
        all_val_labels = []
        all_val_predictions = []
        
        with torch.no_grad():
            for tls, pstats, flowstats, labels in val_loader:
                outputs = model(tls.to(device), pstats.to(device), flowstats.to(device))
                loss = criterion(outputs, labels.to(device))
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                all_val_labels.extend(labels.cpu().numpy())
                all_val_predictions.extend(predicted.cpu().numpy())
        
        val_f1_score_epoch = f1_score(all_val_labels, all_val_predictions, average='macro')
        val_losses.append(val_loss / len(val_loader))
        val_f1_scores.append(val_f1_score_epoch)
        
        end_epoch_time = time()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Time: {(end_epoch_time-start_epoch_time)/60:.2f} min,  Loss: {running_loss/len(train_loader):.4f}, Train F-score: {train_f1_score_epoch:.4f}")
        print(f"Validation Loss: {val_loss/len(val_loader):.4f}, Validation F-score: {val_f1_score_epoch:.4f}")
        
        # Save the model if the validation loss is the best we've seen so far.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            print(f"Saved model with val_loss: {val_loss/len(val_loader):.4f}")
    
        # Plot the training and validation loss curves
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(train_f1_scores, label='Train F-score')
        plt.plot(val_f1_scores, label='Validation F-score')
        plt.xlabel('Epochs')
        plt.ylabel('F-score')
        plt.title('Training and Validation F-score')
        plt.legend()
        
        plt.savefig(plt_save_path)


        #plt.show()
    

In [ ]:
# Initialize model, loss function, and optimizer
num_classes = len(mapdict)
model = UWTrafficClassificationModel(num_classes).to(device)
#model = nn.DataParallel(model) 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
folder_path = ''
model_path = folder_path + 'best_UW_model_cuda.pth'
plt_save_path = folder_path + 'UW_train_validation_plot_cuda.png'

# Train the model
num_epochs = 10
train_UW_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_path, plt_save_path)

In [ ]:
# Test the model
model_path = folder_path + 'best_UW_model_cuda.pth'

num_classes = len(mapdict)

# Initialize the model
model = UWTrafficClassificationModel(num_classes).to(device)

model.load_state_dict(torch.load(model_path))
model.eval()
test_loss = 0.0
all_labels = []
all_predictions = []
        
with torch.no_grad():
    correct = 0
    total = 0
    for tls, pstats, flowstats, labels in test_loader:
        outputs = model(tls.to(device), pstats.to(device), flowstats.to(device))
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.to(device)).sum().item()
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

print(f"Accuracy: {100 * correct / total:.2f}%")

In [ ]:
from classification_reports_UW import classification_reports_average, feature_importance_average, classfication_report_df_save, classfication_report

In [ ]:
classfication_report(all_labels, all_predictions, mapdict)